In [18]:
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, Dataset
from torchvision import datasets, models, transforms
from PIL import Image
import pandas as pd
from pathlib import Path
import os
import random
import numpy as np
from torch.backends import cudnn

RANDOM_STATE = 22 

def set_seed(seed=RANDOM_STATE):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    
    if torch.cuda.is_available():
        torch.cuda.manual_seed(seed)
        torch.cuda.manual_seed_all(seed)
        cudnn.deterministic = True
        cudnn.benchmark = False
    
    if torch.backends.mps.is_available():
        torch.use_deterministic_algorithms(True, warn_only=True)
    
    os.environ['PYTHONHASHSEED'] = str(seed)
    
    
set_seed(RANDOM_STATE)    

device = torch.device("cuda" if torch.cuda.is_available() else "mps" if torch.backends.mps.is_available() else "cpu")
print(f"Используем устройство: {device}")

DATA_DIR = Path("../data")  
TRAIN_DIR = DATA_DIR / "train"
TEST_DIR = DATA_DIR / "test"
SUBMISSION_PATH = DATA_DIR / "sample_submission.csv"

IMAGENET_MEAN = [0.485, 0.456, 0.406]
IMAGENET_STD = [0.229, 0.224, 0.225]

BATCH_SIZE = 8

# АУГМЕНТАЦИИ 
train_transforms = transforms.Compose([
    transforms.RandomResizedCrop(224, scale=(0.85, 1.0), ratio=(0.95, 1.05)),
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.RandomRotation(degrees=20),
    transforms.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.1, hue=0.02),
    transforms.ToTensor(),
    transforms.Normalize(mean=IMAGENET_MEAN, std=IMAGENET_STD),
])

val_transforms = transforms.Compose([
    transforms.Resize(256),
    transforms.CenterCrop(224),
    transforms.ToTensor(),
    transforms.Normalize(mean=IMAGENET_MEAN, std=IMAGENET_STD),
])

# МОДЕЛЬ (ResNet18 с заморозкой)
def create_resnet18_frozen(num_classes=2):
    model = models.resnet18(weights=models.ResNet18_Weights.DEFAULT)
    
    for param in model.parameters():
        param.requires_grad = False
    
    model.fc = nn.Sequential(
        nn.Dropout(0.5),
        nn.Linear(model.fc.in_features, num_classes)
    )
    
    return model


# ЗАГРУЗКА ЛУЧШЕЙ МОДЕЛИ ИЗ K-FOLD
# Загружаем полный датасет (нужен для проверки классов)
full_dataset = datasets.ImageFolder(root=TRAIN_DIR, transform=train_transforms)

# Создаём модель
model = create_resnet18_frozen(num_classes=2).to(device)

# Загружаем веса из best Fold
model_path = "resnet_fold1.pth"

if os.path.exists(model_path):
    model.load_state_dict(torch.load(model_path, map_location=device))
    model.eval()
    print(f"Модель загружена из {model_path}")
else:
    print(f"Файл {model_path} не найден!")
    print("Доступные .pth файлы:", [f for f in os.listdir('.') if f.endswith('.pth')])

# Проверка порядка классов
print(f"\nПорядок классов:")
print(f"   full_dataset.classes = {full_dataset.classes}")
print(f"   full_dataset.class_to_idx = {full_dataset.class_to_idx}")

print("ИНФЕРЕНС НА ТЕСТОВЫХ ДАННЫХ")

class TestImageDataset(Dataset):
    def __init__(self, csv_path, img_dir, transform=None):
        self.df = pd.read_csv(csv_path)
        self.img_dir = Path(img_dir)
        self.transform = transform
    
    def __len__(self):
        return len(self.df)
    
    def __getitem__(self, idx):
        img_id = str(self.df.iloc[idx]['id']).zfill(4)
        img_path = self.img_dir / f"{img_id}.jpg"
        if not img_path.exists():
            img_path = self.img_dir / f"{img_id}.png"
        
        image = Image.open(img_path).convert('RGB')
        if self.transform:
            image = self.transform(image)
        
        return image, img_id

test_dataset = TestImageDataset(
    csv_path=SUBMISSION_PATH,
    img_dir=TEST_DIR,
    transform=val_transforms
)

test_loader = DataLoader(test_dataset, batch_size=32, shuffle=False, num_workers=0)

# Mapping классов (правильный, как мы проверили)
idx_to_class = {0: 'cleaned', 1: 'dirty'}
print(f"\nMapping: {idx_to_class}")

predictions = []
image_ids = []

print("Делаем предсказания...")
with torch.no_grad():
    for images, ids in test_loader:
        images = images.to(device)
        outputs = model(images)
        _, predicted = torch.max(outputs, 1)
        
        for img_id, pred_idx in zip(ids, predicted.cpu().numpy()):
            image_ids.append(img_id)
            predictions.append(idx_to_class[pred_idx])

print(f"Сделано {len(predictions)} предсказаний")

# СОЗДАНИЕ SUBMISSION.CSV
submission_df = pd.DataFrame({
    'id': image_ids,
    'label': predictions
})

submission_path = DATA_DIR / "submission_fold1.csv"
submission_df.to_csv(submission_path, index=False)

print(f"\nSubmission сохранён: {submission_path}")
print(f"\nРаспределение классов:")
print(submission_df['label'].value_counts())

total = len(submission_df)
cleaned = (submission_df['label'] == 'cleaned').sum()
dirty = (submission_df['label'] == 'dirty').sum()

print(f"\nСтатистика:")
print(f"   Всего изображений: {total}")
print(f"   Cleaned: {cleaned} ({cleaned/total*100:.1f}%)")
print(f"   Dirty: {dirty} ({dirty/total*100:.1f}%)")

Используем устройство: mps
Модель загружена из resnet_fold1.pth

Порядок классов:
   full_dataset.classes = ['cleaned', 'dirty']
   full_dataset.class_to_idx = {'cleaned': 0, 'dirty': 1}
ИНФЕРЕНС НА ТЕСТОВЫХ ДАННЫХ

Mapping: {0: 'cleaned', 1: 'dirty'}
Делаем предсказания...
Сделано 744 предсказаний

Submission сохранён: ../data/submission_fold1.csv

Распределение классов:
label
dirty      628
cleaned    116
Name: count, dtype: int64

Статистика:
   Всего изображений: 744
   Cleaned: 116 (15.6%)
   Dirty: 628 (84.4%)


In [15]:
# Проверяем все 5 фолдов
fold_files = [
    "resnet_fold1.pth",
    "resnet_fold2.pth",
    "resnet_fold3.pth",
    "resnet_fold4.pth",
    "resnet_fold5.pth"
]

results = {}

for fold_file in fold_files:
    if not os.path.exists(fold_file):
        print(f"{fold_file} не найден, пропускаем")
        continue
        
    print(f"Тестируем {fold_file}")    
     
    # Загружаем модель
    model = create_resnet18_frozen(num_classes=2).to(device)
    model.load_state_dict(torch.load(fold_file, map_location=device))
    model.eval()
    
    # Делаем предсказания
    predictions = []
    image_ids = []
    
    with torch.no_grad():
        for images, ids in test_loader:
            images = images.to(device)
            outputs = model(images)
            _, predicted = torch.max(outputs, 1)
            
            for img_id, pred_idx in zip(ids, predicted.cpu().numpy()):
                image_ids.append(img_id)
                predictions.append(idx_to_class[pred_idx])
    
    # Считаем распределение
    submission_df = pd.DataFrame({'id': image_ids, 'label': predictions})
    cleaned_pct = (submission_df['label'] == 'cleaned').mean() * 100
    dirty_pct = (submission_df['label'] == 'dirty').mean() * 100
    
    results[fold_file] = {
        'cleaned': cleaned_pct,
        'dirty': dirty_pct,
        'submission': submission_df
    }
    
    print(f"Cleaned: {cleaned_pct:.1f}% | Dirty: {dirty_pct:.1f}%")

# Сводная таблица
print("СВОДНАЯ ТАБЛИЦА ПО ВСЕМ ФОЛДАМ")
print(f"{'Фолд':<20} {'Cleaned %':<12} {'Dirty %':<12}")
for fold_file, stats in results.items():
    print(f"{fold_file:<20} {stats['cleaned']:<12.1f} {stats['dirty']:<12.1f}")

Тестируем resnet_fold1.pth
Cleaned: 15.6% | Dirty: 84.4%
Тестируем resnet_fold2.pth
Cleaned: 14.7% | Dirty: 85.3%
Тестируем resnet_fold3.pth
Cleaned: 5.4% | Dirty: 94.6%
Тестируем resnet_fold4.pth
Cleaned: 36.0% | Dirty: 64.0%
Тестируем resnet_fold5.pth
Cleaned: 76.9% | Dirty: 23.1%
СВОДНАЯ ТАБЛИЦА ПО ВСЕМ ФОЛДАМ
Фолд                 Cleaned %    Dirty %     
resnet_fold1.pth     15.6         84.4        
resnet_fold2.pth     14.7         85.3        
resnet_fold3.pth     5.4          94.6        
resnet_fold4.pth     36.0         64.0        
resnet_fold5.pth     76.9         23.1        


### Лучший результат на Kaggle:  0.73924 на фолде 1

Попробовать ансамбль фолдов